### Testing the nodal --> modal code 2D

Same idea as the 1D code.

In [1]:
import numpy as np
import matplotlib.pylab as plt
import sys
sys.path.append("../..")

In [ ]:
from src.mesh import build_uniform_mesh_2d
from src.testing.helpers_2d import random_modal_coeffs_2d
from src.grid import local_cell_center_nodes_1d
from src.transforms import vandermonde_1d, modal_to_nodal_2d, nodal_to_modal_2d

In [3]:
# setup
xmin, xmax = -1, 1
ymin, ymax = -1, 1
xlim = (xmin, xmax)
ylim = (ymin, ymax)

# mesh
Kx, Ky = 16, 16
p = 3
order = p + 1
mesh = build_uniform_mesh_2d(Kx, Ky, p=p, xlim=xlim, ylim=ylim)

#### Begin with the modal --> nodal --> modal roundtrip test ####
# generate random DG modal coefficients
coeffs_init = random_modal_coeffs_2d(Kx, Ky, p=p, seed=62, scale=1.0)

# choose cell centers as evaluation points (for nodal --> modal, need exactly p + 1 points per element)
n_eval = order
eval_nodes = local_cell_center_nodes_1d(nloc=n_eval)

# evaluate the random modal coeffiecients at nodal locations
# the modal --> nodal code requires a dictionary with vandermonde matrix and modal coefficients atleast
V, Vinv = vandermonde_1d(nodes=eval_nodes, p=p)
dg_init = {
    "coeffs": coeffs_init, 
    "V": V
}

_, Unode = modal_to_nodal_2d(dg_init, return_blocks=True)

# transform this result back into modal coefficients
dg_result = nodal_to_modal_2d(Unode=Unode, mesh=mesh, p=p)

# compare coefficient errors:
coeffs_result = dg_result["coeffs"]

coeffs_err = coeffs_init - coeffs_result

coeff_err_rel_inf = np.max(np.abs(coeffs_err)) / np.max(np.abs(coeffs_init))
coeff_err_rel_l2 = np.linalg.norm(coeffs_err.ravel()) / np.linalg.norm(coeffs_init.ravel())

print("Relative coefficient error (inf) =", coeff_err_rel_inf)
print("Relative coefficient error (l2)  =", coeff_err_rel_l2)

Relative coefficient error (inf) = 3.025682708188964e-16
Relative coefficient error (l2)  = 2.736539721409289e-16


In [4]:
#### nodal --> modal --> nodal rountrip test

_, Unode_recovered = modal_to_nodal_2d(dg_result, return_blocks=True)

nodal_err = Unode - Unode_recovered
nodal_err_max = np.max(np.abs(nodal_err))
nodal_err_rel_l2 = np.linalg.norm(nodal_err.ravel()) / np.linalg.norm(Unode.ravel())

print("Max nodal reconstruction error =", nodal_err_max)
print("Relative nodal l2 error        =", nodal_err_rel_l2)

Max nodal reconstruction error = 1.7763568394002505e-15
Relative nodal l2 error        = 2.500762375815845e-16
